# Notebook 2: Train Continual SD-LoRA Adapter

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


In [ ]:
# Notebook 2 calisma kimligi
# Once bu hucreyi duzenleyin, sonra bootstrap hucresini yeniden calistirin.
CROP_NAME = "tomato"
PART_NAME = "unspecified"
ENABLE_BAYESIAN_OPTIMIZATION = False  # Bayesian optimization devre disi
print(f"[RUN] crop={CROP_NAME} part={PART_NAME} bayes_opt={ENABLE_BAYESIAN_OPTIMIZATION}")


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell03_runtime_setup.py', globals())


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 2 parametreleri ---
    # Bu hucreyi duzenleyin, sonra kalan hucreleri sirayla calistirin.
    # Kosu kimligi icin CROP_NAME/PART_NAME degerlerini ustteki hucreden yonetin.

    # RUNTIME_DATASET_ROOT: Notebook 0'un yazdigi <dataset_key>/continual|val|test|ood yapisini tutan repo-ici root.
    RUNTIME_DATASET_ROOT = "data/prepared_runtime_datasets"

    # DATASET_NAME: Notebook 0'un urettigi runtime dataset klasor adi. Bos ise notebook kullaniciya sorar.
    DATASET_NAME = ""

    # OOD_ROOT: Gercek OOD klasoru. Bos ise ASK_FOR_OOD_ROOT=True iken notebook yol sorar; Enter varsa runtime ood/ kullanir.
    OOD_ROOT = ""
    ASK_FOR_OOD_ROOT = True

    # CROP_NAME ve PART_NAME, kosu adlandirmasi ve metadata icin kullanilir.
    CROP_NAME = globals().get("CROP_NAME", "tomato")
    PART_NAME = globals().get("PART_NAME", "unspecified")
    ENABLE_BAYESIAN_OPTIMIZATION = bool(globals().get("ENABLE_BAYESIAN_OPTIMIZATION", False))

    # ALLOW_UNDER_MIN_TRAINING: True olursa 100 image/class production guardrail'i research kosulari icin bypass edilir.
    ALLOW_UNDER_MIN_TRAINING = False

    # EPOCHS: train split uzerinden kac tam gecis yapilacagi.
    EPOCHS = 12

    # BATCH_SIZE: optimizer adimi basina ornek sayisi. GPU limitine yakin olacak sekilde artirilabilir.
    BATCH_SIZE = 96

    # LEARNING_RATE: adapter/LoRA parametreleri icin optimizer adim buyuklugu.
    LEARNING_RATE = 2e-4

    # LORA_R: LoRA rank degeri. Buyudukce kapasite ve VRAM/islem maliyeti artar.
    LORA_R = 24

    # LORA_ALPHA: LoRA olcekleme katsayisi. Genelde LORA_R degerinin 2x-4x araliginda kullanilir.
    LORA_ALPHA = 24

    # LORA_DROPOUT: LoRA katmanlarina uygulanan dropout. Buyudukce regularizasyon artar.
    LORA_DROPOUT = 0.1

    # OOD_FACTOR: OOD esik hassasiyetini carpansal olarak ayarlar.
    OOD_FACTOR = 3.0
    SURE_SEMANTIC_PERCENTILE = 90.0
    SURE_CONFIDENCE_PERCENTILE = 97.0
    CONFORMAL_ALPHA = 0.05
    CONFORMAL_METHOD = "raps"
    CONFORMAL_RAPS_LAMBDA = 0.2
    CONFORMAL_RAPS_K_REG = 1

    # BER_ENABLED: eski/yeni sinif enerji ayrimi icin deneysel egitim regularizeri.
    BER_ENABLED = False

    # BER_LAMBDA_OLD / BER_LAMBDA_NEW: eski ve yeni sinif kisimlari icin BER ceza agirliklari.
    BER_LAMBDA_OLD = 0.1
    BER_LAMBDA_NEW = 0.1
    BER_WARMUP_STEPS = 50

    # WEIGHT_DECAY: AdamW weight decay degeri.
    WEIGHT_DECAY = 0.01

    # MIXED_PRECISION: {'off', 'auto', 'fp16', 'bf16'} seceneklerinden biri.
    MIXED_PRECISION = "bf16"

    # GRAD_ACCUM_STEPS: gradient accumulation katsayisi.
    GRAD_ACCUM_STEPS = 1

    # MAX_GRAD_NORM: gradient clipping esigi. 0 olursa clipping kapanir.
    MAX_GRAD_NORM = 1.0

    # LABEL_SMOOTHING: CE label smoothing katsayisi.
    LABEL_SMOOTHING = 0.0

    # LOSS_NAME / LOGITNORM_TAU: varsayilan kayip LogitNorm'dur; CE icin loss_name='cross_entropy' secin.
    LOSS_NAME = "logitnorm"
    LOGITNORM_TAU = 1.0

    # Scheduler ayarlari.
    SCHEDULER_NAME = "cosine"
    SCHEDULER_WARMUP_RATIO = 0.1
    SCHEDULER_MIN_LR = 1e-6

    # Early stopping ayarlari.
    EARLY_STOPPING_PATIENCE = 5
    EARLY_STOPPING_MIN_DELTA = 0.0

    # Tekrarlanabilirlik ve runtime ayarlari.
    DETERMINISTIC = False
    TF32_ENABLED = True
    SEED = 42

    # NUM_WORKERS: dataloader worker sayisi. CPU veri yukleme paralelligini belirler.
    NUM_WORKERS = 12

    # PREFETCH: NUM_WORKERS > 0 iken worker basina prefetch katsayisi.
    PREFETCH = 8

    # PIN_MEMORY: host memory sabitleyerek host-to-GPU aktarimini hizlandirir.
    PIN_MEMORY = True

    # USE_CACHE: destekleniyorsa decode edilmis ornekleri RAM'de tutar.
    USE_CACHE = True

    # CACHE_TRAIN_SPLIT: continual/train split'ini de cache'ler. Yuksek RAM'li Colab icin uygundur.
    CACHE_TRAIN_SPLIT = True

    # VALIDATION_EVERY_N_EPOCHS: her N epoch'ta tam validation calistirir; final epoch her zaman dahildir.
    VALIDATION_EVERY_N_EPOCHS = 2

    # CHECKPOINT_EVERY_N_STEPS / CHECKPOINT_ON_EXCEPTION: notebook checkpoint sikligi ayarlari.
    CHECKPOINT_EVERY_N_STEPS = 250
    CHECKPOINT_ON_EXCEPTION = True

    # STDOUT_BATCH_INTERVAL: canli training ilerleme yazdirma araligi.
    STDOUT_BATCH_INTERVAL = 12

    # RESUME_MODE: "fresh" yeni kosu baslatir, "resume" son checkpointten devam eder.
    RESUME_MODE = "fresh"  # "fresh" or "resume"

    # AUTO_DISCONNECT_RUNTIME: tum final exportlar basariliysa Colab runtime'i kapatir.
    AUTO_DISCONNECT_RUNTIME = True

    # AUTO_DISCONNECT_GRACE_SECONDS: son durum gorunsun diye disconnect oncesi kisa bekleme suresi.
    AUTO_DISCONNECT_GRACE_SECONDS = 20

    # AUTO_PUSH_TO_GITHUB: final exportlar bitince runs/<crop>/<part>/<RUN_ID>/ klasorunu repoya commit edip pushlar.
    AUTO_PUSH_TO_GITHUB = True

    # AUTO_PUSH_REMOTE_NAME: auto-push aciksa kullanilacak git remote adi.
    AUTO_PUSH_REMOTE_NAME = "origin"

    # AUTO_PUSH_BRANCH: auto-push icin branch override degeri. None olursa mevcut branch kullanilir.
    AUTO_PUSH_BRANCH = None

    # MANUAL_PARAM_OVERRIDES: anahtarlar notebook degisken adlariyla birebir ayni olmali.
    # Ornek: {"BATCH_SIZE": 32, "EPOCHS": 16, "PIN_MEMORY": False}
    MANUAL_PARAM_OVERRIDES = {}

    # Notebook icinde daha sonra kullanilacak gizli repo varsayimlarini yukle.
    BASE_CONFIG = ConfigurationManager(config_dir=str(ROOT / "config"), environment="colab").load_all_configs()
    CONTINUAL_DATA_CFG = BASE_CONFIG.get("training", {}).get("continual", {}).get("data", {})

    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = bool(TF32_ENABLED)

    TARGET_SIZE = int(CONTINUAL_DATA_CFG.get("target_size", 224))
    DATA_SAMPLER = str(CONTINUAL_DATA_CFG.get("sampler", "shuffle"))
    AUGMENTATION_POLICY = str(CONTINUAL_DATA_CFG.get("augmentation_policy", "randaugment")).strip().lower()
    RANDAUGMENT_NUM_OPS = int(CONTINUAL_DATA_CFG.get("randaugment_num_ops", 2))
    RANDAUGMENT_MAGNITUDE = int(CONTINUAL_DATA_CFG.get("randaugment_magnitude", 7))
    ALLOW_UNDER_MIN_TRAINING = bool(ALLOW_UNDER_MIN_TRAINING)
    LOADER_ERROR_POLICY = str(CONTINUAL_DATA_CFG.get("loader_error_policy", "tolerant"))
    CACHE_SIZE = int(CONTINUAL_DATA_CFG.get("cache_size", 1000))
    VALIDATE_IMAGES_ON_INIT = bool(CONTINUAL_DATA_CFG.get("validate_images_on_init", True))

    from scripts.colab_training_recommendations import resolve_effective_notebook_params

    def _collect_notebook_base_params():
        return {
            "ALLOW_UNDER_MIN_TRAINING": bool(ALLOW_UNDER_MIN_TRAINING),
            "EPOCHS": int(EPOCHS),
            "BATCH_SIZE": int(BATCH_SIZE),
            "LEARNING_RATE": float(LEARNING_RATE),
            "LORA_R": int(LORA_R),
            "LORA_ALPHA": int(LORA_ALPHA),
            "LORA_DROPOUT": float(LORA_DROPOUT),
            "OOD_FACTOR": float(OOD_FACTOR),
            "SURE_SEMANTIC_PERCENTILE": float(SURE_SEMANTIC_PERCENTILE),
            "SURE_CONFIDENCE_PERCENTILE": float(SURE_CONFIDENCE_PERCENTILE),
            "CONFORMAL_ALPHA": float(CONFORMAL_ALPHA),
            "CONFORMAL_METHOD": str(CONFORMAL_METHOD),
            "CONFORMAL_RAPS_LAMBDA": float(CONFORMAL_RAPS_LAMBDA),
            "CONFORMAL_RAPS_K_REG": int(CONFORMAL_RAPS_K_REG),
            "BER_ENABLED": bool(BER_ENABLED),
            "BER_LAMBDA_OLD": float(BER_LAMBDA_OLD),
            "BER_LAMBDA_NEW": float(BER_LAMBDA_NEW),
            "BER_WARMUP_STEPS": int(BER_WARMUP_STEPS),
            "RANDAUGMENT_NUM_OPS": int(RANDAUGMENT_NUM_OPS),
            "RANDAUGMENT_MAGNITUDE": int(RANDAUGMENT_MAGNITUDE),
            "WEIGHT_DECAY": float(WEIGHT_DECAY),
            "MIXED_PRECISION": str(MIXED_PRECISION),
            "GRAD_ACCUM_STEPS": int(GRAD_ACCUM_STEPS),
            "MAX_GRAD_NORM": float(MAX_GRAD_NORM),
            "LABEL_SMOOTHING": float(LABEL_SMOOTHING),
            "LOSS_NAME": str(LOSS_NAME),
            "LOGITNORM_TAU": float(LOGITNORM_TAU),
            "SCHEDULER_NAME": str(SCHEDULER_NAME),
            "SCHEDULER_WARMUP_RATIO": float(SCHEDULER_WARMUP_RATIO),
            "SCHEDULER_MIN_LR": float(SCHEDULER_MIN_LR),
            "EARLY_STOPPING_PATIENCE": int(EARLY_STOPPING_PATIENCE),
            "EARLY_STOPPING_MIN_DELTA": float(EARLY_STOPPING_MIN_DELTA),
            "DETERMINISTIC": bool(DETERMINISTIC),
            "SEED": int(SEED),
            "NUM_WORKERS": int(NUM_WORKERS),
            "PREFETCH": int(PREFETCH),
            "PIN_MEMORY": bool(PIN_MEMORY),
            "USE_CACHE": bool(USE_CACHE),
            "CACHE_TRAIN_SPLIT": bool(CACHE_TRAIN_SPLIT),
            "VALIDATION_EVERY_N_EPOCHS": int(VALIDATION_EVERY_N_EPOCHS),
            "CHECKPOINT_EVERY_N_STEPS": int(CHECKPOINT_EVERY_N_STEPS),
            "CHECKPOINT_ON_EXCEPTION": bool(CHECKPOINT_ON_EXCEPTION),
            "STDOUT_BATCH_INTERVAL": int(STDOUT_BATCH_INTERVAL),
            "RESUME_MODE": str(RESUME_MODE),
            "AUTO_DISCONNECT_RUNTIME": bool(AUTO_DISCONNECT_RUNTIME),
            "AUTO_DISCONNECT_GRACE_SECONDS": int(AUTO_DISCONNECT_GRACE_SECONDS),
            "AUTO_PUSH_TO_GITHUB": bool(AUTO_PUSH_TO_GITHUB),
            "AUTO_PUSH_REMOTE_NAME": str(AUTO_PUSH_REMOTE_NAME),
            "AUTO_PUSH_BRANCH": AUTO_PUSH_BRANCH,
            "ENABLE_BAYESIAN_OPTIMIZATION": bool(ENABLE_BAYESIAN_OPTIMIZATION),
        }

    def _collect_bayesian_notebook_overrides():
        if not bool(ENABLE_BAYESIAN_OPTIMIZATION):
            return {}
        recommendations_path = ROOT / "runs" / "_index" / "bayesian_recommendations.json"
        if not recommendations_path.exists():
            print(f"[BAYES] Toggle acik ama dosya yok: {recommendations_path}")
            return {}
        try:
            payload = json.loads(recommendations_path.read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"[BAYES] Oneri dosyasi okunamadi: {exc}")
            return {}

        cohorts = list(payload.get("cohorts", [])) if isinstance(payload, dict) else []
        selected_cohort = None
        for cohort in cohorts:
            comparability = cohort.get("comparability", {}) if isinstance(cohort, dict) else {}
            if str(comparability.get("crop_name", "")).strip().lower() == str(CROP_NAME).strip().lower() and str(comparability.get("part_name", "")).strip().lower() == str(PART_NAME).strip().lower():
                selected_cohort = cohort
                break
        if selected_cohort is None and cohorts:
            selected_cohort = cohorts[0]

        proposals = list((selected_cohort or {}).get("proposals", [])) if isinstance(selected_cohort, dict) else []
        if not proposals:
            print("[BAYES] Uygulanabilir oneriler bulunamadi.")
            return {}

        proposal = proposals[0]
        parameters = proposal.get("parameters", {}) if isinstance(proposal, dict) else {}
        if not isinstance(parameters, dict):
            return {}

        mapping = {
            "training.learning_rate": "LEARNING_RATE",
            "training.weight_decay": "WEIGHT_DECAY",
            "training.num_epochs": "EPOCHS",
            "training.batch_size": "BATCH_SIZE",
            "training.adapter.lora_r": "LORA_R",
            "training.adapter.lora_alpha": "LORA_ALPHA",
            "training.adapter.lora_dropout": "LORA_DROPOUT",
            "training.ood.threshold_factor": "OOD_FACTOR",
            "training.optimization.logitnorm_tau": "LOGITNORM_TAU",
            "training.data.randaugment_num_ops": "RANDAUGMENT_NUM_OPS",
            "training.data.randaugment_magnitude": "RANDAUGMENT_MAGNITUDE",
        }
        overrides = {}
        for source_key, target_key in mapping.items():
            if source_key in parameters:
                overrides[target_key] = parameters[source_key]

        rank = int(proposal.get("rank", 1)) if str(proposal.get("rank", "")).strip() else 1
        print(f"[BAYES] rank={rank} ile {len(overrides)} parametre onerisi yuklendi.")
        return overrides

    bayesian_overrides = _collect_bayesian_notebook_overrides()
    resolved_manual_overrides = dict(bayesian_overrides)
    resolved_manual_overrides.update(dict(MANUAL_PARAM_OVERRIDES or {}))

    INITIAL_EFFECTIVE_PARAMS = resolve_effective_notebook_params(
        _collect_notebook_base_params(),
        {"recommended_params": _collect_notebook_base_params()},
        resolved_manual_overrides,
        accepted=False,
    )

    STATE = {
        "validated": False,
        "class_names": [],
        "runtime_dataset_root": None,
        "runtime_dataset_key": None,
        "selected_dataset_name": None,
        "selected_dataset_root": None,
        "resolved_ood_root": None,
        "dataset_inspection": {},
        "hardware_inspection": {},
        "recommendation_report": {},
        "recommendation_decision": "pending",
        "effective_params": dict(INITIAL_EFFECTIVE_PARAMS),
        "adapter": None,
        "loaders": None,
        "history": None,
        "calibration": None,
        "checkpoint_manager": CHECKPOINT_MANAGER,
        "resume_manifest": None,
        "best_val_loss": None,
        "best_metric_state": {},
        "auto_disconnect_report": None,
        "git_push_report": None,
        "telemetry": TELEMETRY,
    }

    if str(INITIAL_EFFECTIVE_PARAMS.get("RESUME_MODE", RESUME_MODE)) == "resume":
        try:
            STATE["resume_manifest"] = CHECKPOINT_MANAGER.get_latest()
            if STATE["resume_manifest"]:
                manifest = STATE["resume_manifest"]
                print(
                    f"[RESUME] Checkpoint bulundu: {manifest.get('name', '?')} "
                    f"epoch={manifest.get('epoch', 0)} step={manifest.get('global_step', 0)}"
                )
        except Exception:
            pass

    hf_token_ready = False
    hf_token = str(resolve_hf_token() or "").strip()
    if not hf_token:
        raise RuntimeError(
            "Notebook 2 egitim baslamadan once Hugging Face token ister. "
            "HF_TOKEN, HUGGINGFACE_TOKEN veya HUGGINGFACE_HUB_TOKEN degerini env var ya da Colab secret olarak tanimlayin."
        )
    hf_token_ready = bool(login_and_check_hf_token(print_fn=print))
    if not hf_token_ready:
        raise RuntimeError(
            "Hugging Face token on kontrolu basarisiz oldu. Egitimden once tokeni duzeltin."
        )

    github_token_ready = False
    if bool(INITIAL_EFFECTIVE_PARAMS.get("AUTO_PUSH_TO_GITHUB", AUTO_PUSH_TO_GITHUB)):
        github_token = str(resolve_github_token() or "").strip()
        if not github_token:
            raise RuntimeError(
                "AUTO_PUSH_TO_GITHUB is enabled, but no GitHub token was found. "
                "Egitim baslamadan once GH_TOKEN veya GITHUB_TOKEN degerini env var ya da Colab secret olarak tanimlayin."
            )
        github_token_ready = True
        print("[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.")

    print(
        f"[CONFIG] source=notebook_cell crop={CROP_NAME} epochs={EPOCHS} bs={BATCH_SIZE} "
        f"lr={LEARNING_RATE} resume={RESUME_MODE} ber={BER_ENABLED} "
        f"loss={LOSS_NAME} tau={LOGITNORM_TAU} auto_disconnect={AUTO_DISCONNECT_RUNTIME} "
        f"auto_push={AUTO_PUSH_TO_GITHUB} bayes_opt={ENABLE_BAYESIAN_OPTIMIZATION}"
    )
    print(
        f"[DATASET] runtime_root={RUNTIME_DATASET_ROOT} dataset_name={DATASET_NAME or '<ask>'}"
    )
    print(
        f"[OOD] ood_root={OOD_ROOT or '<ask>'} ask_for_ood_root={ASK_FOR_OOD_ROOT}"
    )
    print(
        f"[RUNTIME] defaults=notebook_cell mp={MIXED_PRECISION} workers={NUM_WORKERS} prefetch={PREFETCH} "
        f"sched={SCHEDULER_NAME} wd={WEIGHT_DECAY} accum={GRAD_ACCUM_STEPS} grad_clip={MAX_GRAD_NORM} "
        f"label_smooth={LABEL_SMOOTHING} warmup={SCHEDULER_WARMUP_RATIO} "
        f"early_stop={EARLY_STOPPING_PATIENCE}/{EARLY_STOPPING_MIN_DELTA} "
        f"val_every={VALIDATION_EVERY_N_EPOCHS} cache_train={CACHE_TRAIN_SPLIT} "
        f"aug={AUGMENTATION_POLICY} randaug={RANDAUGMENT_NUM_OPS}/{RANDAUGMENT_MAGNITUDE} "
    )
    print(
        f"[OOD] factor={OOD_FACTOR} sure={SURE_SEMANTIC_PERCENTILE}/{SURE_CONFIDENCE_PERCENTILE} "
        f"conformal={CONFORMAL_METHOD} alpha={CONFORMAL_ALPHA} "
        f"raps_lambda={CONFORMAL_RAPS_LAMBDA} raps_k={CONFORMAL_RAPS_K_REG}"
    )
    TELEMETRY.update_latest(
        {
            "phase": "parameters_ready",
            "parameter_source": "notebook_cell",
            "crop": CROP_NAME,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "dataset_name": DATASET_NAME,
            "ood_root": OOD_ROOT,
            "ask_for_ood_root": ASK_FOR_OOD_ROOT,
            "loss_name": LOSS_NAME,
            "logitnorm_tau": LOGITNORM_TAU,
            "mixed_precision": MIXED_PRECISION,
            "num_workers": NUM_WORKERS,
            "prefetch": PREFETCH,
            "cache_train_split": CACHE_TRAIN_SPLIT,
            "augmentation_policy": AUGMENTATION_POLICY,
            "randaugment_num_ops": RANDAUGMENT_NUM_OPS,
            "randaugment_magnitude": RANDAUGMENT_MAGNITUDE,
            "allow_under_min_training": ALLOW_UNDER_MIN_TRAINING,
            "ber_enabled": BER_ENABLED,
            "ber_lambda_old": BER_LAMBDA_OLD,
            "ber_lambda_new": BER_LAMBDA_NEW,
            "ber_warmup_steps": BER_WARMUP_STEPS,
            "ood_factor": OOD_FACTOR,
            "sure_semantic_percentile": SURE_SEMANTIC_PERCENTILE,
            "sure_confidence_percentile": SURE_CONFIDENCE_PERCENTILE,
            "conformal_alpha": CONFORMAL_ALPHA,
            "conformal_method": CONFORMAL_METHOD,
            "conformal_raps_lambda": CONFORMAL_RAPS_LAMBDA,
            "conformal_raps_k_reg": CONFORMAL_RAPS_K_REG,
            "label_smoothing": LABEL_SMOOTHING,
            "scheduler_warmup_ratio": SCHEDULER_WARMUP_RATIO,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
            "resume_mode": RESUME_MODE,
            "validation_every_n_epochs": VALIDATION_EVERY_N_EPOCHS,
            "enable_bayesian_optimization": bool(ENABLE_BAYESIAN_OPTIMIZATION),
            "auto_push_to_github": AUTO_PUSH_TO_GITHUB,
            "hf_token_ready": hf_token_ready,
            "auto_push_token_ready": github_token_ready,
            "auto_push_remote_name": AUTO_PUSH_REMOTE_NAME,
            "auto_push_branch": AUTO_PUSH_BRANCH,
            "auto_disconnect_runtime": AUTO_DISCONNECT_RUNTIME,
            "auto_disconnect_grace_seconds": AUTO_DISCONNECT_GRACE_SECONDS,
        }
    )


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell05_access_check.py', globals())


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Validation"):
    from scripts.colab_dataset_layout import list_repo_dataset_directories, resolve_direct_repo_dataset_root, resolve_repo_relative_root
    from scripts.colab_training_recommendations import (
        inspect_runtime_dataset,
        inspect_runtime_hardware,
        recommend_notebook_training_params,
        resolve_effective_notebook_params,
    )

    crop_key = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(CROP_NAME).strip())
    while "__" in crop_key:
        crop_key = crop_key.replace("__", "_")
    crop_key = crop_key.strip("_")
    if not crop_key:
        raise RuntimeError("CROP_NAME bos olmayan bir crop anahtarina cozulmeli.")

    runtime_parent = resolve_repo_relative_root(repo_root=ROOT, repo_relative_root=RUNTIME_DATASET_ROOT)
    direct_runtime_dataset = resolve_direct_repo_dataset_root(
        repo_root=ROOT,
        repo_relative_root=RUNTIME_DATASET_ROOT,
    )
    runtime_dirs = [] if direct_runtime_dataset is not None else list_repo_dataset_directories(
        repo_root=ROOT,
        repo_relative_root=RUNTIME_DATASET_ROOT,
    )
    candidates = []
    if direct_runtime_dataset is not None:
        direct_runtime_name, direct_runtime_path = direct_runtime_dataset
        candidates.append({"name": direct_runtime_name, "path": direct_runtime_path, "parent": runtime_parent})
    else:
        candidates.extend(
            {"name": path.name, "path": path, "parent": runtime_parent}
            for path in runtime_dirs
        )
    if not candidates:
        raise RuntimeError("No prepared runtime datasets were found under RUNTIME_DATASET_ROOT. Notebook 0'u once calistirin.")

    requested_dataset_name = str(DATASET_NAME).strip()
    if requested_dataset_name:
        matches = [item for item in candidates if item["name"] == requested_dataset_name]
        if not matches:
            available_options = [item['name'] for item in candidates]
            raise RuntimeError(
                f"Requested dataset '{requested_dataset_name}' was not found. Available options: {available_options}"
            )
        selected = matches[0]
    elif len(candidates) == 1:
        selected = candidates[0]
        print(f"[DATASET] Yalnizca bir runtime dataset bulundu, otomatik secildi: {selected['name']}")
    else:
        print("[DATASET] Kullanilabilir runtime dataset secenekleri:")
        for index, item in enumerate(candidates, start=1):
            print(f"  [{index}] {item['name']} ({item['path']})")
        raw_choice = str(input(f"Kullanilacak dataset icin numara ya da isim girin (1-{len(candidates)}): ")).strip()
        if not raw_choice:
            raise RuntimeError("Dataset secimi bos birakilamaz.")
        if raw_choice.isdigit():
            selected_index = int(raw_choice) - 1
            if selected_index < 0 or selected_index >= len(candidates):
                raise RuntimeError(f"Dataset secim index'i aralik disi: {raw_choice}")
            selected = candidates[selected_index]
        else:
            matches = [item for item in candidates if item['name'] == raw_choice]
            if not matches:
                raise RuntimeError(f"Dataset secimi bulunamadi: {raw_choice}")
            selected = matches[0]

    selected_dataset_name = str(selected['name'])
    selected_dataset_root = Path(selected['path'])
    if not selected_dataset_name.startswith(crop_key):
        raise RuntimeError(
            f"Secilen runtime dataset CROP_NAME ile uyusmuyor: {selected_dataset_name} vs {crop_key}"
        )
    missing_splits = [name for name in ("continual", "val", "test") if not (selected_dataset_root / name).is_dir()]
    if missing_splits:
        raise RuntimeError(f"Prepared runtime dataset is missing split folder(s): {missing_splits}")
    class_names = sorted(d.name for d in (selected_dataset_root / "continual").iterdir() if d.is_dir())
    if not class_names:
        raise RuntimeError(f"No class subdirectories in prepared runtime split: {selected_dataset_root / 'continual'}")
    runtime_root = selected_dataset_root.parent
    default_ood_root = selected_dataset_root / "ood"
    requested_ood_root = str(OOD_ROOT or "").strip()
    if bool(ASK_FOR_OOD_ROOT) and not requested_ood_root:
        default_hint = str(default_ood_root) if default_ood_root.is_dir() else ""
        prompt = "OOD klasoru yolunu girin"
        if default_hint:
            prompt += f" [Enter={default_hint}]"
        requested_ood_root = str(input(prompt + ": ")).strip()
        if not requested_ood_root and default_hint:
            requested_ood_root = default_hint

    if requested_ood_root:
        resolved_ood_root = Path(requested_ood_root).expanduser()
        if not resolved_ood_root.is_absolute():
            resolved_ood_root = (ROOT / resolved_ood_root).resolve()
        if not resolved_ood_root.is_dir():
            raise RuntimeError(f"OOD klasoru bulunamadi veya klasor degil: {resolved_ood_root}")
        print(f"[OOD] explicit ood root={resolved_ood_root}")
        resolved_ood_root_value = str(resolved_ood_root)
    elif default_ood_root.is_dir():
        print(f"[OOD] runtime ood root={default_ood_root}")
        resolved_ood_root_value = str(default_ood_root)
    else:
        print("[OOD] Gercek OOD split secilmedi; fallback held-out benchmark kullanilabilir.")
        resolved_ood_root_value = ""
    print(f"[DATASET] runtime root={selected_dataset_root} classes={len(class_names)}: {class_names}")

    base_params = _collect_notebook_base_params()
    dataset_inspection = inspect_runtime_dataset(selected_dataset_root, ood_root=resolved_ood_root_value or None)
    hardware_inspection = inspect_runtime_hardware(DEVICE)
    recommendation_report = recommend_notebook_training_params(base_params, dataset_inspection, hardware_inspection)

    split_totals = dict(dataset_inspection.get("split_totals", {}))
    print(
        f"[RECOMMEND][DATASET] scale={dataset_inspection.get('dataset_scale_bucket', 'unknown')} "
        f"continual={split_totals.get('continual', 0)} val={split_totals.get('val', 0)} "
        f"test={split_totals.get('test', 0)} ood={split_totals.get('ood', 0)} "
        f"classes={dataset_inspection.get('class_count', 0)}"
    )
    print(
        f"[RECOMMEND][HW] device={hardware_inspection.get('effective_device', 'cpu')} "
        f"gpu={hardware_inspection.get('gpu_name') or 'none'} "
        f"vram_gb={hardware_inspection.get('total_vram_gb')} cpu_count={hardware_inspection.get('cpu_count', 0)}"
    )

    blockers = list(recommendation_report.get("blockers", []))
    warnings = list(recommendation_report.get("warnings", []))
    for item in warnings:
        print(f"[RECOMMEND][WARN] {item}")
    for item in blockers:
        print(f"[RECOMMEND][BLOCK] {item}")

    accepted_recommendations = False
    recommendation_decision = "no_changes"
    if recommendation_report.get("has_changes"):
        print("[RECOMMEND] Onerilen degisiklikler:")
        for key in sorted(recommendation_report.get("changes", {})):
            change = recommendation_report["changes"][key]
            print(f"  - {key}: {change['current']} -> {change['recommended']} | {change['reason']}")
        if blockers:
            recommendation_decision = "blocked"
            print("[RECOMMEND] Blocker oldugu icin otomatik uygulama sorusu atlandi. Gerekirse MANUAL_PARAM_OVERRIDES ile acik override girin.")
        else:
            raw_confirm = str(input("Apply recommended parameters? [y/N]: ")).strip().lower()
            accepted_recommendations = raw_confirm in {"y", "yes"}
            recommendation_decision = "accepted" if accepted_recommendations else "rejected"
    elif blockers:
        recommendation_decision = "blocked"
        print("[RECOMMEND] Blocker var; notebook ham parametrelerle devam eder. Gerekirse MANUAL_PARAM_OVERRIDES guncellenmeli.")
    else:
        print("[RECOMMEND] Mevcut notebook parametreleri onerilen degerlerle zaten uyumlu.")

    effective_params = resolve_effective_notebook_params(
        base_params,
        recommendation_report,
        MANUAL_PARAM_OVERRIDES,
        accepted=accepted_recommendations,
    )
    if MANUAL_PARAM_OVERRIDES:
        print(f"[RECOMMEND] Manual overrides uygulandi: {MANUAL_PARAM_OVERRIDES}")
    print(
        f"[RECOMMEND][FINAL] epochs={effective_params['EPOCHS']} bs={effective_params['BATCH_SIZE']} "
        f"lr={effective_params['LEARNING_RATE']} lora={effective_params['LORA_R']}/{effective_params['LORA_ALPHA']} "
        f"dropout={effective_params['LORA_DROPOUT']} accum={effective_params['GRAD_ACCUM_STEPS']} "
        f"workers={effective_params['NUM_WORKERS']} prefetch={effective_params['PREFETCH']}"
    )

    STATE["class_names"] = class_names
    STATE["validated"] = True
    STATE["runtime_dataset_root"] = runtime_root
    STATE["runtime_dataset_key"] = selected_dataset_name
    STATE["selected_dataset_name"] = selected_dataset_name
    STATE["selected_dataset_root"] = selected_dataset_root
    STATE["resolved_ood_root"] = resolved_ood_root_value
    STATE["dataset_inspection"] = dataset_inspection
    STATE["hardware_inspection"] = hardware_inspection
    STATE["recommendation_report"] = recommendation_report
    STATE["recommendation_decision"] = recommendation_decision
    STATE["effective_params"] = effective_params
    TELEMETRY.update_latest(
        {
            "phase": "dataset_validated",
            "dataset_root": str(selected_dataset_root),
            "runtime_dataset_root": str(runtime_root),
            "runtime_dataset_key": selected_dataset_name,
            "selected_dataset_name": selected_dataset_name,
            "resolved_ood_root": resolved_ood_root_value,
            "class_count": len(class_names),
            "recommendation_decision": recommendation_decision,
            "recommendation_change_count": int(recommendation_report.get('change_count', 0)),
        }
    )


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell07_engine_init.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell08_ood_config_verify.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell09_training.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell10_ood_calibration.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell11_adapter_save.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell12_final_evaluation.py', globals())
